# FCC gasoline desulfurization (S Zorb)

`difflow_refinery.desulfurization.SZorbUnit` is a reactive-adsorption desulfurizer. It removes sulfur per class, carries the sorbent's loading round a circulating loop, and saturates some olefins, which costs octane. Every rate and sorbent constant is **illustrative**: the unit lands in the published ranges for S Zorb but reproduces no plant data. This notebook:

1. runs the base case;
2. traces the sulfur-octane trade-off across severity;
3. shows the sorbent loop saturating as feed sulfur rises;
4. differentiates through the loop's root solve.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
from difflow_refinery.desulfurization import SZorbFeed, SZorbUnit

jax.config.update("jax_enable_x64", True)

def feed(s=500.0):
    return SZorbFeed.from_properties(50.0, sg=0.73, sulfur_wppm=s, olefins_vol=30.0,
                                     aromatics_vol=22.0, ron=92.0, mon=80.0)

unit = SZorbUnit()
out = unit.solve(feed())
for k in ("product.S_wppm", "product.RON", "ron.loss", "h2.consumption_wt", "hds.conversion",
          "sorbent.activity", "balance.mass", "balance.S"):
    print(f"{k:22s} {float(out[k]):.4g}")

product.S_wppm         7.917
product.RON            91.12
ron.loss               0.8797
h2.consumption_wt      0.06843
hds.conversion         0.9842
sorbent.activity       0.6032
balance.mass           -2.22e-16
balance.S              0


## The sulfur-octane trade-off

Raising severity, by lowering the space velocity or heating the bed, removes more sulfur and saturates more olefins. Olefin saturation has the higher activation energy, so temperature buys sulfur at a steeper octane price than space velocity does.

In [2]:
print("WHSV   S wppm   RON loss")
for w in (8.0, 6.0, 4.0, 3.0, 2.0):
    o = unit.solve(feed(), whsv=w)
    print(f"{w:4.1f}  {float(o['product.S_wppm']):7.2f}  {float(o['ron.loss']):7.3f}")
print("\nT (C)  S wppm   RON loss")
for T in (673.15, 693.15, 713.15):
    o = unit.solve(feed(), T=T)
    print(f"{T - 273.15:5.0f}  {float(o['product.S_wppm']):7.2f}  {float(o['ron.loss']):7.3f}")

WHSV   S wppm   RON loss
 8.0    49.42    0.449


 6.0    26.38    0.595
 4.0     7.92    0.880


 3.0     2.50    1.156
 2.0     0.28    1.686

T (C)  S wppm   RON loss


  400    20.59    0.562
  420     7.92    0.880


  440     2.46    1.331


## The sorbent loop

Captured sulfur loads the circulating sorbent, and a loaded sorbent removes less. As the feed sulfur rises, the activity falls, and product sulfur rises faster than in proportion to the feed.

In [3]:
print("feed S  product S  activity")
for s in (100.0, 300.0, 500.0, 1000.0, 1500.0):
    o = unit.solve(feed(s))
    print(f"{s:6.0f}  {float(o['product.S_wppm']):9.2f}  {float(o['sorbent.activity']):8.3f}")

feed S  product S  activity
   100       0.63     0.760


   300       3.00     0.681
   500       7.92     0.603


  1000      48.82     0.419
  1500     187.61     0.275


## Gradients through the loop

The sorbent loading is the root of one equation, which optimistix differentiates implicitly. The Jacobian below covers product sulfur and RON loss with respect to feed sulfur, temperature, space velocity and circulation.

In [4]:
def f(x):
    o = unit.solve(feed(x[0]), T=x[1], whsv=x[2], circulation=x[3])
    return jnp.stack([o["product.S_wppm"], o["ron.loss"]])

J = jax.jacfwd(f)(jnp.array([500.0, 693.15, 4.0, 0.025]))
print("          d/dS_feed    d/dT      d/dWHSV   d/dcirc")
for name, row in zip(("S wppm", "RON loss"), np.asarray(J)):
    print(f"{name:9s} " + " ".join(f"{v:9.4g}" for v in row))

          d/dS_feed    d/dT      d/dWHSV   d/dcirc
S wppm      0.03408   -0.4195     6.984    -364.8
RON loss         -0   0.01897   -0.2104        -0
